## RMSE, MRE uncertainty
This notebook evaluates the uncertainty of the root mean square error (RMSE) and the mean relative error (MRE) of the neural network prediction by using bootstrap method.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import time

import utilities.plot_settings

In [ ]:
def bootstrap(predictions: np.ndarray, targets: np.ndarray, n_set: int) -> (np.ndarray, np.ndarray):
	 
	boot_rmse = np.zeros(n_set)
	boot_mre = np.zeros(n_set)
	data_indices = np.arange(len(predictions))
	
	for i in range(n_set):

		boot_indices = np.random.choice(data_indices, size=len(predictions), replace=True)
		boot_predictions = predictions[boot_indices]
		boot_targets = targets[boot_indices]		
		boot_rmse[i] = np.sqrt(np.mean((boot_predictions - boot_targets)**2))
		boot_mre[i] = np.mean(abs(boot_predictions - boot_targets)/boot_targets)

	return boot_rmse, boot_mre

In [ ]:
data = pd.read_csv('../../data/paper_results/ronchi_etal_2021/test_set_inference/test_inference_results_sigmak.csv')
par1_target_sigmak = data["target:sigma_k"].to_numpy()
par1_prediction_sigmak = data["predicted:sigma_k"].to_numpy()

data = pd.read_csv('../../data/paper_results/ronchi_etal_2021/test_set_inference/test_inference_results_hc.csv')
par1_target_hc = data["target:h_c"].to_numpy()
par1_prediction_hc = data["predicted:h_c"].to_numpy()

data = pd.read_csv('../../data/paper_results/ronchi_etal_2021/test_set_inference/test_inference_results_sigmak_hc.csv')
par2_target_hc = data["target:h_c"].to_numpy()
par2_target_sigmak = data["target:sigma_k"].to_numpy()
par2_prediction_hc = data["predicted:h_c"].to_numpy()
par2_prediction_sigmak = data["predicted:sigma_k"].to_numpy()

In [ ]:
# bootstrap to obtain un uncertainty over the RMSE and MRE values of the predictions.

par1_boot_rmse_sigmak, par1_boot_mre_sigmak = bootstrap(par1_prediction_sigmak, par1_target_sigmak, 1000)
par1_boot_rmse_hc, par1_boot_mre_hc = bootstrap(par1_prediction_hc, par1_target_hc, 1000)
par2_boot_rmse_sigmak, par2_boot_mre_sigmak = bootstrap(par2_prediction_sigmak, par2_target_sigmak, 1000)
par2_boot_rmse_hc, par2_boot_mre_hc = bootstrap(par2_prediction_hc, par2_target_hc, 1000)

std_par1_boot_rmse_sigmak = np.std(par1_boot_rmse_sigmak)
std_par1_boot_rmse_hc = np.std(par1_boot_rmse_hc)
std_par2_boot_rmse_sigmak = np.std(par2_boot_rmse_sigmak)
std_par2_boot_rmse_hc = np.std(par2_boot_rmse_hc)

print("Relative uncertainty on the RMSE value:")
print(std_par1_boot_rmse_sigmak/np.mean(par1_boot_rmse_sigmak))
print(std_par1_boot_rmse_hc/np.mean(par1_boot_rmse_hc))
print(std_par2_boot_rmse_sigmak/np.mean(par2_boot_rmse_sigmak))
print(std_par2_boot_rmse_hc/np.mean(par2_boot_rmse_hc))

std_par1_boot_mre_sigmak = np.std(par1_boot_mre_sigmak)
std_par1_boot_mre_hc = np.std(par1_boot_mre_hc)
std_par2_boot_mre_sigmak = np.std(par2_boot_mre_sigmak)
std_par2_boot_mre_hc = np.std(par2_boot_mre_hc)

print("Relative uncertainty on the MRE value:")
print(std_par1_boot_mre_sigmak/np.mean(par1_boot_mre_sigmak))
print(std_par1_boot_mre_hc/np.mean(par1_boot_mre_hc))
print(std_par2_boot_mre_sigmak/np.mean(par2_boot_mre_sigmak))
print(std_par2_boot_mre_hc/np.mean(par2_boot_mre_hc))

In [ ]:
rmse_sigmak_edges = np.linspace(0,11.,51)
fig, ax = plt.subplots()
ax.hist(
    par1_boot_rmse_sigmak,
    bins=rmse_sigmak_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
	label=r'1-par. prediction',
)
ax.axvline(x=np.mean(par1_boot_rmse_sigmak), linestyle='--', linewidth=3, color='tab:blue')
ax.hist(
    par2_boot_rmse_sigmak,
    bins=rmse_sigmak_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=0.7,
	label=r'2-par. prediction',
)
ax.axvline(x=np.mean(par2_boot_rmse_sigmak), linestyle='--', linewidth=3, color='tab:red')
ax.set_xlabel(r'RMSE $\sigma_{\rm k}$ [km s$^{-1}$]')
ax.set_ylabel('Number counts')
ax.legend(frameon=False, loc=0, prop={'size': 15})
plt.show(block=False)

In [ ]:
rmse_hc_edges = np.linspace(0,0.05,51)
fig, ax = plt.subplots()
ax.hist(
    par1_boot_rmse_hc,
    bins=rmse_hc_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
	label=r'1-par. prediction',
)
ax.axvline(x=np.mean(par1_boot_rmse_hc), linestyle='--', linewidth=3, color='tab:blue')
ax.hist(
    par2_boot_rmse_hc,
    bins=rmse_hc_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=0.7,
	label=r'2-par. prediction',
)
ax.axvline(x=np.mean(par2_boot_rmse_hc), linestyle='--', linewidth=3, color='tab:red')
ax.set_xlabel(r'RMSE $h_{\rm c}$ [kpc]')
ax.set_ylabel('Number counts')
ax.legend(frameon=False, loc=0, prop={'size': 15})
plt.show(block=False)

In [ ]:
mre_sigmak_edges = np.linspace(0., 0.05,51)
fig, ax = plt.subplots()
ax.hist(
    par1_boot_mre_sigmak,
    bins=mre_sigmak_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
	label=r'1-par. prediction',
)
ax.axvline(x=np.mean(par1_boot_mre_sigmak), linestyle='--', linewidth=3, color='tab:blue')
ax.hist(
    par2_boot_mre_sigmak,
    bins=mre_sigmak_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=0.7,
	label=r'2-par. prediction',
)
ax.axvline(x=np.mean(par2_boot_mre_sigmak), linestyle='--', linewidth=3, color='tab:red')
ax.set_xlabel(r'MRE $\sigma_{\rm k}$')
ax.set_ylabel('Number counts')
ax.legend(frameon=False, loc=0, prop={'size': 15})
plt.show(block=False)

In [ ]:
mre_hc_edges = np.linspace(0,0.1,51)
fig, ax = plt.subplots()
ax.hist(
    par1_boot_mre_hc,
    bins=mre_hc_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=0.7,
	label=r'1-par. prediction',
)
ax.axvline(x=np.mean(par1_boot_mre_hc), linestyle='--', linewidth=3, color='tab:blue')
ax.hist(
    par2_boot_mre_hc,
    bins=mre_hc_edges,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=0.7,
	label=r'2-par. prediction',
)
ax.axvline(x=np.mean(par2_boot_mre_hc), linestyle='--', linewidth=3, color='tab:red')
ax.set_xlabel(r'MRE $h_{\rm c}$')
ax.set_ylabel('Number counts')
ax.legend(frameon=False, loc=0, prop={'size': 15})
plt.show(block=False)

In [ ]:
# plot correlation between the two parameters
fig, ax = plt.subplots()

ax.set_xlabel(r'RMSE $\sigma_{\rm k}$ [km s$^{-1}$]')
ax.set_ylabel(r'RMSE $h_{\rm c}$ [kpc]')

ax.scatter(
    par2_boot_rmse_sigmak,
    par2_boot_rmse_hc,
    linestyle="None",
    marker="o",
    facecolors='black',
    edgecolors='None',
    s=10,
    alpha=0.5,
    rasterized=True,
)

plt.show(block=False)

ig, ax = plt.subplots()

ax.set_xlabel(r'MRE $\sigma_{\rm k}$')
ax.set_ylabel(r'MRE $h_{\rm c}$')

ax.scatter(
    par2_boot_mre_sigmak,
    par2_boot_mre_hc,
    linestyle="None",
    marker="o",
    facecolors='black',
    edgecolors='None',
    s=10,
    alpha=0.5,
    rasterized=True,
)

plt.show(block=False)